# 02 - Limpieza y Preparación de Datos
## Hotel Dann Monasterio - Proyecto de Analítica Descriptiva

## Objetivo del notebook
Limpiar el dataset consolidado producto del notebook `01_exploracion_dataset.ipynb` y dejarlo listo para el análisis exploratorio y descriptivo. Las tareas son:

1. Cargar el dataset consolidado de los huéspedes (Hoja1 + Hoja2).
2. Eliminar columnas con 100% de valores nulos.
3. Eliminar columnas sin variabilidad (un único valor).
4. Anonimizar datos personales (PII) mediante hash SHA-256.
5. Tratar valores atípicos en `edad_aco` (valores fuera de rango plausible).
6. Estandarizar tipos de fechas.
7. Calcular variables derivadas: `duracion_estancia`, `lead_time`, `anio`, `mes`, `dia_semana`, `ingreso_total`, `rango_edad`.
8. Eliminar duplicados.
9. Persistir el dataset limpio en `data/processed/reservas_clean.parquet`.

## Contexto CRISP-DM
Este notebook corresponde a la fase **Preparación de los datos**. Su salida es el insumo de los notebooks 03 (EDA) y 04 (Análisis Descriptivo).

## Importar librerías

In [18]:
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 80)

## Cargar dataset original

Cargamos las dos hojas del Excel y las consolidamos. El resultado es el mismo `df` que se exploró en el notebook 01.

In [19]:
ruta = Path("../data/raw/DataSet_ReservaYHuespedes_V01.xlsx")

# Lee todas las hojas automáticamente
hojas = pd.read_excel(ruta, sheet_name=None)

# Mostrar shape por hoja
for nombre, df_hoja in hojas.items():
    print(f"{nombre}: {df_hoja.shape}")

if len(hojas) == 1:
    df = next(iter(hojas.values())).copy()
else:
    df = pd.concat(hojas.values(), ignore_index=True)

print(f"Consolidado: {df.shape}")
df.head()


Hoja1: (65534, 79)
Hoja2: (5348, 79)
Consolidado: (70882, 79)


,fecha,codigp_pla_reg,consecut,codigp_pla,codsegmento,codmotivacion,ident_aco,tdcto_aco_his,clasi_aco_his,idn_aco_his,codigoclase_his,nrohab_hab_his,foliotitular_his,nfolio_his,codigoprogramacliente,registro,nfolio,nombre_aco,tdcto_aco,fllega_aco,fsalid_aco,porce_aco,vlr_aco,clasi_aco,nrohab_hab,nfolor_aco,idn_aco,estado_aco,adici_aco,edad_aco,sexo_aco,adicmaex,fcheckout,usuchout,numvoucher,descri_pla,nreser_res,codofic,adultos,ninos,adicionales,nombre_emp,tarifa,adicional,nacionalidad,oficio,alimen_pla,incognito,fechasischin,campo1,campo2,campo3,codiga_age,nombre_age,tiphab_tip,complejo,descricomplejo,codigoclase,descripcionclase,vlrus_red,us_reg,orden,descripcionorden,valorplan,ivaplan,servicioplan,valorconsumoadicional,ivaconsumoadicional,servicioconsumoadicional,totalconsumosplan,totalconsumosadicional,codigocategoria,nombrecategoria,clahab_clh,codigotemporada,nombretemporada,folio_titular,folio_maestro,tipofolio
0,2020.06.01,CORP1,114472,CORP1,COR,CCE,1075659763,CE,A,T,NaN,207,NaN,93022.0,NaN,78039,93022.0,ROZO HERNANDEZ DARIO ALEXANDER,CE,2020-06-01,2020-06-02,0,0,A,207,93022.0,T,32,N,36,M,NaN,2020-06-02,JEFERSON,75523,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,183800.0,70000.0,COL,165.0,S,N,2020-06-10 18:56:00.000,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93022,93022,H
1,2020.06.10,CORP1,114473,CORP1,COR,CCE,1075659763,CE,A,T,NaN,207,NaN,93023.0,NaN,78040,93023.0,ROZO HERNANDEZ DARIO ALEXANDER,CE,2020-06-10,2020-06-11,0,0,A,207,93023.0,T,32,N,36,M,NaN,2020-06-11,JEFERSON,75524,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,J RESTREPO EQUIPHOS SAS,183800.0,70000.0,COL,165.0,S,N,2020-06-11 15:29:00.000,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93023,93023,H
2,2020.06.11,CORP1,114474,CORP1,COR,CCE,1112099164,CE,A,T,NaN,207,NaN,93025.0,NaN,78041,93025.0,PEREZ MONTOYA CARLOS AUGUSTO,CE,2020-06-11,2020-06-12,0,0,A,207,93025.0,T,32,N,126,M,NaN,2020-06-12,AESTRADA,75525,CORPORATIVO 1,NaN,NaN,NaN,NaN,NaN,KELLOGG DE COLOMBIA S.A.,183800.0,70000.0,COL,96.0,S,N,2020-06-12 07:16:00.000,NaN,NaN,NaN,NaN,NaN,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,36100.0,0.0,0.0,0.0,0.0,226100.0,0.0,NaN,No Definida,SG,NaN,NaN,93025,93025,H
3,2020.07.02,CORP1,114475,CORP1,COR,CCE,16842997,CE,A,T,NaN,207,NaN,93030.0,NaN,78042,93030.0,GARZON PARRA OSCAR JULIAN,CE,2020-07-02,2020-07-18,0,0,A,207,93030.0,T,32,N,45,M,NaN,2020-07-18,AESTRADA,75526,CORPORATIVO 1,70660.0,2.0,NaN,NaN,NaN,CEMENTOS ARGOS S.A,183800.0,70000.0,COL,165.0,S,N,2020-07-03 08:19:59.999,NaN,NaN,NaN,AVIA,AVIATUR,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,0.0,0.0,0.0,0.0,0.0,190000.0,0.0,NaN,No Definida,SG,NaN,NaN,93030,93030,H
4,2020.07.03,CORP1,114475,CORP1,COR,CCE,16842997,CE,A,T,NaN,207,NaN,93030.0,NaN,78042,93030.0,GARZON PARRA OSCAR JULIAN,CE,2020-07-02,2020-07-18,0,0,A,207,93030.0,T,32,N,45,M,NaN,2020-07-18,AESTRADA,75526,CORPORATIVO 1,70660.0,2.0,NaN,NaN,NaN,CEMENTOS ARGOS S.A,183800.0,70000.0,COL,165.0,S,N,2020-07-03 08:19:59.999,NaN,NaN,NaN,AVIA,AVIATUR,SE,HOTEL,HOTEL DANN MONASTERI,NaN,NaN,0.0,P,NaN,NaN,190000.0,0.0,0.0,0.0,0.0,0.0,190000.0,0.0,NaN,No Definida,SG,NaN,NaN,93030,93030,H


## Crear copia para trabajar

Siempre trabajamos sobre una copia, así conservamos el DataFrame original por si necesitamos volver atrás.

In [20]:
df_clean = df.copy()
print(f"Trabajando sobre copia: {df_clean.shape}")

Trabajando sobre copia: (70882, 79)


## Paso 1 - Eliminar columnas con 100% de valores nulos

In [21]:
nulos_pct = df_clean.isnull().mean() * 100
cols_100_nulas = nulos_pct[nulos_pct == 100].index.tolist()

print(f"Columnas 100% nulas a eliminar: {len(cols_100_nulas)}")
for c in cols_100_nulas:
    print(f"  - {c}")

df_clean = df_clean.drop(columns=cols_100_nulas)
print(f"\nShape después: {df_clean.shape}")

Columnas 100% nulas a eliminar: 9
  - codigoclase_his
  - codigoprogramacliente
  - adicmaex
  - campo1
  - codigoclase
  - descripcionclase
  - orden
  - descripcionorden
  - codigocategoria

Shape después: (70882, 70)


## Paso 2 - Eliminar columnas sin variabilidad (un único valor)

In [22]:
unicos = df_clean.nunique(dropna=True)
cols_un_valor = unicos[unicos == 1].index.tolist()

print(f"Columnas con un único valor a eliminar: {len(cols_un_valor)}")
for c in cols_un_valor:
    print(f"  - {c}: {df_clean[c].dropna().unique()}")

df_clean = df_clean.drop(columns=cols_un_valor)
print(f"\nShape después: {df_clean.shape}")

Columnas con un único valor a eliminar: 10
  - porce_aco: [0]
  - vlr_aco: [0]
  - adici_aco: ['N']
  - ninos: [0.]
  - adicionales: [0.]
  - complejo: ['HOTEL']
  - descricomplejo: ['HOTEL DANN MONASTERI']
  - us_reg: ['P']
  - nombrecategoria: ['No Definida']
  - tipofolio: ['H']

Shape después: (70882, 60)


## Paso 3 - Anonimización de datos personales (PII)

Las siguientes columnas contienen información personal identificable y deben transformarse antes de usarlas en cualquier análisis:

- `ident_aco` (número de documento) → reemplazar por hash SHA-256 truncado a 16 caracteres.
- `nombre_aco` (nombre completo) → eliminar (no aporta al análisis y es PII directa).

**Fundamento legal**: Ley 1581 de 2012 y Decreto 1377 de 2013 (Colombia).

In [23]:
def hash_id(x):
    """Genera un hash SHA-256 truncado a 16 caracteres a partir de un identificador."""
    return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:16]

# 1) Hashear ident_aco
df_clean["id_huesped"] = df_clean["ident_aco"].apply(hash_id)

# 2) Eliminar nombre completo e identificación original
cols_pii = [c for c in ["ident_aco", "nombre_aco"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_pii)

print("Columnas PII eliminadas/transformadas:", cols_pii)
print("\nEjemplo de id_huesped anonimizado:")
df_clean[["id_huesped"]].head(3)

Columnas PII eliminadas/transformadas: ['ident_aco', 'nombre_aco']

Ejemplo de id_huesped anonimizado:


,id_huesped
0,62c5448fe5fff8b2
1,62c5448fe5fff8b2
2,3fb006a62bd449ea


## Paso 4 - Tratamiento de `edad_aco` fuera de rango: imputacion por mediana de segmento

En el notebook 01 se detectaron edades imposibles (0 y edades >= 100). Estos valores son **errores de captura** (digitacion manual incorrecta en el sistema operativo del hotel), no representan huespedes reales fuera de ese rango.

### Estrategia: imputacion por mediana del segmento comercial

En lugar de dejar NaN (lo que perderia informacion demografica para esos registros), se imputa el valor con la **mediana de edad valida del mismo segmento** (`codsegmento`). El razonamiento de negocio es:

- Cada segmento tiene un perfil de edad caracteristico: los huespedes corporativos (COR, CE) tienden a concentrarse entre 35-55 anos, los de tour & travel (T&T) tienen mayor dispersion, los de empresas medianas (EM) corresponden a poblacion laboral activa (25-50 anos).
- Usar la **mediana** en lugar de la media es mas robusto ante la asimetria natural de la distribucion de edades y ante el efecto de los propios outliers que queremos corregir.
- Si un segmento no tuviera suficientes registros validos para calcular su mediana (caso poco probable), se usa la **mediana global** como fallback.

**Politica final:** los registros con edad fuera del rango [1, 99] reciben el valor imputado de la mediana de su segmento. No se elimina ninguna fila; la columna original `edad_aco` se preserva para trazabilidad y se crea `edad_aco_limpia` con el valor corregido.


In [24]:
print('=== Diagnostico ANTES del tratamiento ===')
print(df_clean['edad_aco'].describe().round(2))
print()

# ── 1. Identificar registros fuera de rango ────────────────────────────
rango_valido = df_clean['edad_aco'].between(1, 99)
n_invalidos  = (~rango_valido).sum()
print(f'Registros con edad fuera del rango [1, 99]: {n_invalidos:,}')
print()

# ── 2. Calcular mediana por segmento (solo sobre edades validas) ───────
mediana_por_segmento = (
    df_clean.loc[rango_valido, ['codsegmento', 'edad_aco']]
    .groupby('codsegmento')['edad_aco']
    .median()
    .round(0)
    .astype(int)
)
mediana_global = int(df_clean.loc[rango_valido, 'edad_aco'].median())

print('Mediana de edad por segmento (calculada sobre registros validos):')
for seg, med in mediana_por_segmento.sort_values().items():
    n_seg = (df_clean['codsegmento'] == seg).sum()
    print(f'  {seg:6s}  ->  {med} anos  ({n_seg:,} registros en el segmento)')
print(f'  Mediana global (fallback): {mediana_global} anos')
print()

# ── 3. Imputar: mediana del segmento (o global si el segmento no tiene datos) ─
def imputar_edad(row, mediana_seg, mediana_global):
    if 1 <= row['edad_aco'] <= 99:
        return row['edad_aco']           # edad valida: se conserva tal cual
    seg = row.get('codsegmento', None)
    if seg in mediana_seg.index:
        return mediana_seg[seg]           # mediana del segmento
    return mediana_global                 # fallback: mediana global

df_clean['edad_aco_limpia'] = df_clean.apply(
    imputar_edad,
    axis=1,
    mediana_seg=mediana_por_segmento,
    mediana_global=mediana_global
).astype(int)

# ── 4. Trazabilidad: flag de si el valor fue imputado ──────────────────
df_clean['edad_fue_imputada'] = ~rango_valido

# ── 5. Verificacion post-imputacion ───────────────────────────────────
print('=== Verificacion DESPUES de la imputacion ===')
print(df_clean['edad_aco_limpia'].describe().round(2))
print()
print(f'Edades fuera de rango en edad_aco_limpia: '
      f'{(~df_clean["edad_aco_limpia"].between(1, 99)).sum():,}  <- debe ser 0')
print(f'Registros con edad imputada: {df_clean["edad_fue_imputada"].sum():,}')
print()

# Distribucion de valores imputados por segmento
print('Valores asignados por segmento a los registros imputados:')
imputados = df_clean[df_clean['edad_fue_imputada']]
if len(imputados) > 0:
    resumen = imputados.groupby('codsegmento')['edad_aco_limpia'].agg(['count','first'])
    resumen.columns = ['n_imputados', 'valor_asignado']
    print(resumen.to_string())
else:
    print('  No hay registros con edad imputada.')


=== Diagnostico ANTES del tratamiento ===
count    70882.00
mean        48.07
std         21.80
min          0.00
25%         35.00
50%         46.00
75%         61.00
max        126.00
Name: edad_aco, dtype: float64

Registros con edad fuera del rango [1, 99]: 1,839

Mediana de edad por segmento (calculada sobre registros validos):
  AER     ->  36 anos  (57 registros en el segmento)
  COR     ->  44 anos  (13,144 registros en el segmento)
  EM      ->  44 anos  (18,781 registros en el segmento)
  GRU     ->  45 anos  (428 registros en el segmento)
  TG      ->  46 anos  (9 registros en el segmento)
  ME      ->  46 anos  (21,027 registros en el segmento)
  CE      ->  47 anos  (7,066 registros en el segmento)
  PAR     ->  47 anos  (1,119 registros en el segmento)
  BI      ->  48 anos  (266 registros en el segmento)
  ATN     ->  51 anos  (276 registros en el segmento)
  BG      ->  54 anos  (5 registros en el segmento)
  T&T     ->  58 anos  (8,704 registros en el segmento)
  Media

## Paso 5 - Estandarizar tipos de fechas

Convertimos todas las columnas de fecha a `datetime64` para facilitar las operaciones temporales.

In [25]:
cols_fecha = ["fllega_aco", "fsalid_aco", "fcheckout", "fechasischin"]
for c in cols_fecha:
    if c in df_clean.columns:
        df_clean[c] = pd.to_datetime(df_clean[c], errors="coerce")

# fecha viene como 'AAAA.MM.DD' (string)
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], format="%Y.%m.%d", errors="coerce")

df_clean[cols_fecha + ["fecha"]].dtypes

fllega_aco      datetime64[ns]
fsalid_aco      datetime64[ns]
fcheckout       datetime64[ns]
fechasischin    datetime64[ns]
fecha           datetime64[ns]
dtype: object

## Paso 6 - Corrección de valores nulos y negativos en variables financieras

Antes de calcular totales de ingreso o variables derivadas, es fundamental sanear las variables financieras base.
El análisis exploratorio (notebook 01) identificó dos tipos de problemas: valores negativos y valores nulos
en las variables de consumos adicionales.

### ¿Por qué corregir antes de calcular totales?

- `totalconsumosplan` = `valorplan` + `ivaplan` + `servicioplan`
- `totalconsumosadicional` = `valorconsumoadicional` + `ivaconsumoadicional` + `servicioconsumoadicional`
- `ingreso_total` = `totalconsumosplan` + `totalconsumosadicional` + `tarifa` + `adicional`

Si alguno de los componentes tiene un valor negativo o nulo, el total heredará ese error
y distorsionará los KPIs de negocio.

### Variables y tratamientos aplicados

| Variable | Problema | Tratamiento |
|---|---|---|
| `adicional` | Valores `NaN` (sin recargos registrados) | → `fillna(0)` — no tuvo recargos, válido |
| `valorconsumoadicional` | Valores negativos / `NaN` | → `clip(lower=0)` + `fillna(0)` |
| `servicioconsumoadicional` | Valores negativos / `NaN` | → `clip(lower=0)` + `fillna(0)` |
| `totalconsumosadicional` | Valores negativos / `NaN` | → `clip(lower=0)` + `fillna(0)` |

**Criterio de negocio:** un consumo adicional puede ser $0 (no consumió nada adicional), pero nunca puede
ser negativo, ya que representaría una deuda inexistente del hotel hacia el huésped.
De igual forma, una tarifa negativa no tiene sentido operativo. El valor $0 en `tarifa` corresponde
a cortesías, atenciones institucionales o reservas de empleados sin costo, y es perfectamente válido.


In [26]:
# Variables con problemas identificados en el notebook 01 (EDA)
vars_adicionales = ["valorconsumoadicional", "servicioconsumoadicional", "totalconsumosadicional"]

# --- Diagnóstico ANTES de la corrección ------------------------------------
print("=" * 65)
print("DIAGNÓSTICO ANTES DE LA CORRECCIÓN")
print("=" * 65)

# adicional: solo nulos (no se detectaron negativos en EDA)
if "adicional" in df_clean.columns:
    col_adi = pd.to_numeric(df_clean["adicional"], errors="coerce")
    print(f"  {'adicional':<32} | negativos: {(col_adi < 0).sum():>5,} | nulos: {col_adi.isna().sum():>5,}")

# variables con negativos y nulos
for var in vars_adicionales:
    if var in df_clean.columns:
        col = pd.to_numeric(df_clean[var], errors="coerce")
        print(f"  {var:<32} | negativos: {(col < 0).sum():>5,} | nulos: {col.isna().sum():>5,}")
    else:
        print(f"  {var:<32} | -- columna no existe en el dataset (se omite)")

# --- Corrección 1: castear a numérico ---------------------------------------
todas = ["adicional"] + vars_adicionales
for var in todas:
    if var in df_clean.columns:
        df_clean[var] = pd.to_numeric(df_clean[var], errors="coerce")

# --- Corrección 2: `adicional` — NaN → 0 ------------------------------------
# Sin recargos registrados = 0, no NaN.
# No se corrigen negativos porque no se detectaron en el EDA.
if "adicional" in df_clean.columns:
    n_null = df_clean["adicional"].isna().sum()
    df_clean["adicional"] = df_clean["adicional"].fillna(0)
    print(f"\n  [adicional]  NaN imputados a 0: {n_null:,} registros")

# --- Corrección 3: consumos adicionales — NaN → 0 y negativos → 0 -----------
print()
for var in vars_adicionales:
    if var in df_clean.columns:
        n_neg  = (df_clean[var] < 0).sum()
        n_null = df_clean[var].isna().sum()
        df_clean[var] = df_clean[var].fillna(0).clip(lower=0)
        print(f"  [{var}]")
        print(f"      negativos corregidos a 0 : {n_neg:,}")
        print(f"      nulos imputados a 0       : {n_null:,}")

# --- Verificación DESPUÉS ---------------------------------------------------
print()
print("=" * 65)
print("VERIFICACIÓN DESPUÉS DE LA CORRECCIÓN")
print("=" * 65)
for var in todas:
    if var not in df_clean.columns:
        continue
    col    = df_clean[var]
    n_neg  = (col < 0).sum()
    n_null = col.isna().sum()
    ok = "OK" if (n_neg == 0 and n_null == 0) else "REVISAR"
    print(f"  [{ok}] {var:<32} | negativos: {n_neg:>5,} | nulos: {n_null:>5,}")


DIAGNÓSTICO ANTES DE LA CORRECCIÓN
  adicional                        | negativos:     0 | nulos:   304
  valorconsumoadicional            | negativos:     1 | nulos:     0
  servicioconsumoadicional         | negativos:     1 | nulos:     0
  totalconsumosadicional           | negativos:     1 | nulos:     0

  [adicional]  NaN imputados a 0: 304 registros

  [valorconsumoadicional]
      negativos corregidos a 0 : 1
      nulos imputados a 0       : 0
  [servicioconsumoadicional]
      negativos corregidos a 0 : 1
      nulos imputados a 0       : 0
  [totalconsumosadicional]
      negativos corregidos a 0 : 1
      nulos imputados a 0       : 0

VERIFICACIÓN DESPUÉS DE LA CORRECCIÓN
  [OK] adicional                        | negativos:     0 | nulos:     0
  [OK] valorconsumoadicional            | negativos:     0 | nulos:     0
  [OK] servicioconsumoadicional         | negativos:     0 | nulos:     0
  [OK] totalconsumosadicional           | negativos:     0 | nulos:     0


## Paso 7 - Variables derivadas

Calculamos columnas nuevas a partir de las fechas y los ingresos. Estas variables serán usadas en el notebook 03 y en el diagrama estrella.

In [27]:
# --- Duracion de la estancia (noches programadas) -------------------------
# fsalid_aco - fllega_aco = salida programada - llegada programada.
# Ambas fechas se fijan al hacer la reserva y representan las noches
# contratadas (metrica central de revenue management).
df_clean["duracion_estancia"] = (
    df_clean["fsalid_aco"] - df_clean["fllega_aco"]
).dt.days

# Validacion: duracion negativa = fechas invertidas (error de captura)
n_neg  = (df_clean["duracion_estancia"] < 0).sum()
n_cero = (df_clean["duracion_estancia"] == 0).sum()
print(f"duracion_estancia — negativos (se anulan): {n_neg:,}  |  ceros: {n_cero:,}")
df_clean.loc[df_clean["duracion_estancia"] < 0, "duracion_estancia"] = None

# --- Lead time (dias de anticipacion de la reserva) ------------------------
# 'fecha' = fecha en que se realizo la reserva en el sistema
# 'fllega_aco' = fecha de llegada programada (check-in)
# lead_time = cuantos dias antes de llegar se hizo la reserva
# Se normalizan ambas a fecha pura para evitar ruido por componente horaria.
df_clean["lead_time"] = (
    df_clean["fllega_aco"].dt.normalize() -
    df_clean["fecha"].dt.normalize()
).dt.days

# Negativos = reserva registrada despues de la llegada (error). Se ajustan a 0.
n_lt_neg = (df_clean["lead_time"] < 0).sum()
print(f"lead_time — negativos (se ajustan a 0): {n_lt_neg:,}")
df_clean.loc[df_clean["lead_time"] < 0, "lead_time"] = 0

# --- Variables temporales derivadas de la fecha de llegada ------------------
df_clean["anio"]       = df_clean["fllega_aco"].dt.year
df_clean["mes"]        = df_clean["fllega_aco"].dt.month
df_clean["trimestre"]  = df_clean["fllega_aco"].dt.quarter
df_clean["dia_semana"] = df_clean["fllega_aco"].dt.day_name()

# --- Periodo COVID ----------------------------------------------------------
def periodo_covid(anio):
    if anio in (2020, 2021):
        return "Pandemia"
    if anio == 2022:
        return "Recuperacion"
    return "Post-pandemia"

df_clean["periodo_covid"] = df_clean["anio"].apply(periodo_covid)

# --- Ingreso total ----------------------------------------------------------
# Formula completa segun jerarquia financiera del negocio:
#   totalconsumosplan      = valorplan + ivaplan + servicioplan
#   totalconsumosadicional = valorconsumoadicional + ivaconsumoadicional
#                            + servicioconsumoadicional
#   ingreso_total          = totalconsumosplan + totalconsumosadicional
#                            + tarifa + adicional
df_clean["ingreso_total"] = (
    pd.to_numeric(df_clean["totalconsumosplan"],      errors="coerce").fillna(0).clip(lower=0) +
    pd.to_numeric(df_clean["totalconsumosadicional"], errors="coerce").fillna(0) +
    pd.to_numeric(df_clean["tarifa"],                 errors="coerce").fillna(0).clip(lower=0) +
    pd.to_numeric(df_clean["adicional"],              errors="coerce").fillna(0)
).round(2)

# --- Rango de edad para analisis demografico --------------------------------
bins   = [0, 18, 25, 35, 50, 65, 100]
labels = ["<18", "18-25", "26-35", "36-50", "51-65", "66+"]
df_clean["rango_edad"] = pd.cut(df_clean["edad_aco_limpia"], bins=bins, labels=labels)

# --- Vista previa -----------------------------------------------------------
print(f"\nduracion_estancia — media: {df_clean['duracion_estancia'].mean():.1f} noches  "
      f"max: {df_clean['duracion_estancia'].max():.0f}")
print(f"lead_time         — media: {df_clean['lead_time'].mean():.1f} dias  "
      f"max: {df_clean['lead_time'].max():.0f}")
print(f"ingreso_total     — min: {df_clean['ingreso_total'].min():,.0f}  "
      f"max: {df_clean['ingreso_total'].max():,.0f}  "
      f"media: {df_clean['ingreso_total'].mean():,.0f}")

df_clean[[
    "duracion_estancia", "lead_time", "anio", "mes",
    "dia_semana", "periodo_covid", "ingreso_total", "rango_edad"
]].head()


duracion_estancia — negativos (se anulan): 0  |  ceros: 0
lead_time — negativos (se ajustan a 0): 31,203

duracion_estancia — media: 4.1 noches  max: 183
lead_time         — media: 0.0 dias  max: 0
ingreso_total     — min: 0  max: 4,329,559  media: 434,930


,duracion_estancia,lead_time,anio,mes,dia_semana,periodo_covid,ingreso_total,rango_edad
0,1.0,0,2020,6,Monday,Pandemia,479900.0,36-50
1,1.0,0,2020,6,Wednesday,Pandemia,479900.0,36-50
2,1.0,0,2020,6,Thursday,Pandemia,479900.0,36-50
3,16.0,0,2020,7,Thursday,Pandemia,443800.0,36-50
4,16.0,0,2020,7,Thursday,Pandemia,443800.0,36-50


### Validar la duración de estancia

Esperamos valores positivos. Si hay registros con `fsalid_aco < fllega_aco`, son errores que se marcan.

In [28]:
neg = (df_clean["duracion_estancia"] < 0).sum()
ceros = (df_clean["duracion_estancia"] == 0).sum()
print(f"Registros con duración negativa: {neg:,}")
print(f"Registros con duración = 0 días: {ceros:,}")
print(f"\nDistribución de duración de estancia:")
print(df_clean["duracion_estancia"].describe())

Registros con duración negativa: 0
Registros con duración = 0 días: 0

Distribución de duración de estancia:
count    70882.000000
mean         4.142222
std         11.565632
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        183.000000
Name: duracion_estancia, dtype: float64


## Paso 8 - Reemplazar infinitos y manejar nulos

Estandarizamos `inf`/`-inf` a NaN para evitar errores en cálculos posteriores.

In [29]:
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Tabla resumen final de nulos por columna
resumen = pd.DataFrame({
    "nulos": df_clean.isnull().sum(),
    "% nulos": (df_clean.isnull().mean()*100).round(2)
}).sort_values("% nulos", ascending=False)
resumen.head(15)

,nulos,% nulos
adultos,70866,99.98
campo3,70672,99.70
campo2,70637,99.65
oficio,68303,96.36
foliotitular_his,45149,63.70
nombretemporada,37760,53.27
codigotemporada,37760,53.27
nfolio_his,25733,36.30
nfolio,25726,36.29
nfolor_aco,25726,36.29


## Paso 9 - Eliminar duplicados

Buscamos duplicados exactos (todas las columnas idénticas). Por construcción del PMS no esperamos muchos.

In [30]:
dup = df_clean.duplicated().sum()
print(f"Duplicados exactos: {dup:,}")
df_clean = df_clean.drop_duplicates()
print(f"Shape final tras drop_duplicates: {df_clean.shape}")

Duplicados exactos: 0
Shape final tras drop_duplicates: (70882, 70)


## Paso 10 - Tratamiento de valores atípicos en `nacionalidad`

Durante el EDA (notebook 01) se detectaron registros donde la columna `nacionalidad` contiene **códigos numéricos** en lugar de nombres de país. Estos códigos pertenecen al sistema de identificación de países utilizado por la **DIAN (Dirección de Impuestos y Aduanas Nacionales de Colombia)** para sus procesos de importación, exportación y declaración de información exógena.

La DIAN asigna a cada país un código numérico propio (distinto al estándar ISO 3166-1). Cuando el sistema del hotel registró la reserva, en algunos casos capturó el código DIAN en lugar del nombre textual del país, creando una inconsistencia en el campo.

### Códigos DIAN detectados en el dataset

| Código DIAN | País           |
|:-----------:|:---------------|
| 063         | ARGENTINA      |
| 072         | AUSTRIA        |
| 149         | CANADÁ         |
| 169         | COLOMBIA       |
| 244         | ANGOLA         |
| 247         | UZBEKISTÁN     |

### Estrategia
Se construye un **diccionario de mapeo** con los 6 códigos identificados y se reemplaza el código numérico por el nombre del país correspondiente. Los valores que ya contienen un nombre textual no se modifican.

In [31]:
# ── Paso 10: Corrección de códigos DIAN en columna 'nacionalidad' ──────────

# Mapeo oficial DIAN → nombre de país
# Fuente: Lista de países DIAN para información exógena y comercio exterior
DIAN_CODIGOS = {
    "063": "ARG",
    "072": "AUS",
    "149": "CAN",
    "169": "COL",
    "244": "ANG",
    "247": "UZB",
    # Agregar nuevos códigos aquí si el EDA detecta más
}

# Diagnóstico ANTES
print("=" * 60)
print("DIAGNÓSTICO ANTES — valores únicos en 'nacionalidad'")
print("=" * 60)
nat_antes = df_clean["nacionalidad"].astype(str).str.strip()
codigos_presentes = nat_antes[nat_antes.isin(DIAN_CODIGOS.keys())]
print(f"Registros con código DIAN detectado : {len(codigos_presentes):,}")
print(codigos_presentes.value_counts().to_string())

# Normalizar: strip + upper para comparación consistente
df_clean["nacionalidad"] = (
    df_clean["nacionalidad"]
    .astype(str)
    .str.strip()
    .replace(DIAN_CODIGOS)   # reemplaza solo los códigos exactos del dict
)

# También capturar variante con decimal si vino de Excel (ej. '169.0')
# (ya está cubierta con las claves '169.0' en el dict)

# Diagnóstico DESPUÉS
print("\n" + "=" * 60)
print("DIAGNÓSTICO DESPUÉS — verificación de corrección")
print("=" * 60)
nat_despues = df_clean["nacionalidad"]
codigos_residuales = nat_despues.astype(str).str.strip().isin(DIAN_CODIGOS.keys())
print(f"Códigos DIAN residuales (deben ser 0) : {codigos_residuales.sum():,}")
print(f"\nTop 15 valores más frecuentes en 'nacionalidad' tras corrección:")
print(nat_despues.value_counts().head(15).to_string())


DIAGNÓSTICO ANTES — valores únicos en 'nacionalidad'
Registros con código DIAN detectado : 805
nacionalidad
149    357
169    270
063    164
072      8
244      5
247      1

DIAGNÓSTICO DESPUÉS — verificación de corrección
Códigos DIAN residuales (deben ser 0) : 0

Top 15 valores más frecuentes en 'nacionalidad' tras corrección:
nacionalidad
COL    48731
USA     3520
FRA     2740
ALE     1625
ITA     1400
HOL     1321
MEX     1234
SUI     1149
ESP     1001
BLG      941
ECU      777
CAN      504
REU      474
PER      418
CHI      412


## Paso 11 - Imputación de valores por defecto en campos nulos

En esta etapa se normalizan campos críticos para evitar llaves vacías en el modelo dimensional
y garantizar mapeo consistente en los ETL de dimensiones y hechos.

Se imputa cuando el valor viene nulo o vacío (incluye cadenas en blanco).

Reglas de imputación:

1. Temporada:
- `codigotemporada` -> `ND`
- `nombretemporada` -> `Sin temporada registrada.`

2. Canal:
- `codiga_age` -> `ND`
- `nombre_age` -> `Sin canal registrado`

3. Empresa:
- `nombre_emp` -> `Sin empresa registrada`

Estas reglas permiten que todos los registros tengan un valor de referencia válido
y evitan pérdidas por fallas de mapeo en claves foráneas.


In [32]:
# ── Paso 11: Imputación de valores por defecto en campos nulos/vacíos ─────────

def imputar_nulo_vacio(df, col, valor_default):
    if col not in df.columns:
        print(f"[WARN] Columna no existe, se omite: {col}")
        return 0

    # Normalizar a string para evaluar vacíos de forma uniforme
    serie = df[col].astype("string")
    mascara = serie.isna() | (serie.str.strip() == "")

    n = int(mascara.sum())
    df.loc[mascara, col] = valor_default
    return n


print("=" * 70)
print("IMPUTACIÓN DE CAMPOS CLAVE (TEMPORADA, CANAL, EMPRESA)")
print("=" * 70)

# 1) Temporada
n_cod_temp = imputar_nulo_vacio(df_clean, "codigotemporada", "ND")
n_nom_temp = imputar_nulo_vacio(df_clean, "nombretemporada", "Sin temporada registrada.")

# 2) Canal
n_cod_canal = imputar_nulo_vacio(df_clean, "codiga_age", "ND")
n_nom_canal = imputar_nulo_vacio(df_clean, "nombre_age", "Sin canal registrado")

# 3) Empresa
n_nom_emp = imputar_nulo_vacio(df_clean, "nombre_emp", "Sin empresa registrada")

print(f"codigotemporada imputados : {n_cod_temp:,}")
print(f"nombretemporada imputados : {n_nom_temp:,}")
print(f"codiga_age imputados      : {n_cod_canal:,}")
print(f"nombre_age imputados      : {n_nom_canal:,}")
print(f"nombre_emp imputados      : {n_nom_emp:,}")

print("\nVerificación rápida (nulos remanentes):")
for c in ["codigotemporada", "nombretemporada", "codiga_age", "nombre_age", "nombre_emp"]:
    if c in df_clean.columns:
        nulos = int(df_clean[c].isna().sum())
        vacios = int((df_clean[c].astype("string").str.strip() == "").sum())
        print(f"  {c:<15} -> nulos: {nulos:,} | vacíos: {vacios:,}")

IMPUTACIÓN DE CAMPOS CLAVE (TEMPORADA, CANAL, EMPRESA)
codigotemporada imputados : 37,760
nombretemporada imputados : 37,760
codiga_age imputados      : 8,842
nombre_age imputados      : 8,843
nombre_emp imputados      : 17,505

Verificación rápida (nulos remanentes):
  codigotemporada -> nulos: 0 | vacíos: 0
  nombretemporada -> nulos: 0 | vacíos: 0
  codiga_age      -> nulos: 0 | vacíos: 0
  nombre_age      -> nulos: 0 | vacíos: 0
  nombre_emp      -> nulos: 0 | vacíos: 0


## Paso 12 - Selección final de variables del proyecto

Para los notebooks siguientes (modelado dimensional, machine learning y visualización) solo necesitamos las columnas depuradas y derivadas. Descartamos columnas intermedias (e.g. `edad_aco_limpia`) que ya cumplieron su función.

In [33]:
VARS_FINAL = [
    # Tiempo
    # Fechas clave (fecha = fecha de reserva, necesaria para lead_time)
    "fecha", "fllega_aco", "fsalid_aco", "fechasischin", "fcheckout",
    # Variables temporales derivadas
    "anio", "mes", "trimestre", "dia_semana", "periodo_covid",
    
    # Segmento/Motivación
    "codsegmento", "codmotivacion",

    # Plan / Tarifa
    "codigp_pla", "descri_pla", "alimen_pla", "tarifa", "adicional",

    # Temporada
    "codigotemporada", "nombretemporada",

    # Canal / Agencia
    "codiga_age", "nombre_age", "nombre_emp",

    # Habitación
    "tiphab_tip", "clahab_clh", "nrohab_hab",

    # Huésped
        # Identificador anonimizado
    "id_huesped", 
    "edad_aco", "edad_aco_limpia", "edad_fue_imputada", "rango_edad",
    "sexo_aco", "nacionalidad", "clasi_aco", "idn_aco", "incognito",

    # Ingresos
        # Plan
    "valorplan", "ivaplan", "servicioplan", "totalconsumosplan",
        # Consumo adicional
    "valorconsumoadicional", "ivaconsumoadicional", "servicioconsumoadicional", "totalconsumosadicional",

    # Folio / Reserva
    "folio_titular",
    
    # Ingreso total de la reserva.
    "ingreso_total",
    # Variables derivadas de estancia
    "duracion_estancia", "lead_time",
]

# Filtrar solo las columnas que existen en df_clean
VARS_FINAL = [v for v in VARS_FINAL if v in df_clean.columns]
df_proyecto = df_clean[VARS_FINAL].copy()

print(f"Variables seleccionadas : {len(VARS_FINAL)}")
print(f"Shape df_proyecto       : {df_proyecto.shape}")
print("\nColumnas finales:")
for v in VARS_FINAL:
    dtype = str(df_proyecto[v].dtype)
    nulos = df_proyecto[v].isna().sum()
    print(f"  {v:<35} {dtype:<15} nulos: {nulos:,}")


Variables seleccionadas : 47
Shape df_proyecto       : (70882, 47)

Columnas finales:
  fecha                               datetime64[ns]  nulos: 0
  fllega_aco                          datetime64[ns]  nulos: 0
  fsalid_aco                          datetime64[ns]  nulos: 0
  fechasischin                        datetime64[ns]  nulos: 0
  fcheckout                           datetime64[ns]  nulos: 0
  anio                                int32           nulos: 0
  mes                                 int32           nulos: 0
  trimestre                           int32           nulos: 0
  dia_semana                          object          nulos: 0
  periodo_covid                       object          nulos: 0
  codsegmento                         object          nulos: 0
  codmotivacion                       object          nulos: 0
  codigp_pla                          object          nulos: 0
  descri_pla                          object          nulos: 0
  alimen_pla                    

## Paso 13 - Exportar dataset limpio

Generamos dos formatos de salida a partir de `df_proyecto`:
- **Parquet** – para consumo desde los scripts del modelo dimensional.
- **Excel** – para revisión manual y validación del equipo.

In [34]:
import subprocess, sys
from pathlib import Path

# Instalar pyarrow si no está disponible
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Sanitizar dtypes antes de exportar ------------------------------------
# pyarrow falla con tipos no estándar de pandas (Categorical, Period, bool).
# Se convierten a tipos básicos compatibles sin perder el valor.
def sanitize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        dtype_name = str(df[col].dtype)
        if dtype_name == "category":
            df[col] = df[col].astype(str)          # rango_edad: "<18", "18-25", ...
        elif dtype_name == "bool":
            df[col] = df[col].astype("Int8")        # edad_fue_imputada: True/False -> 1/0
        elif "period" in dtype_name.lower():
            df[col] = df[col].astype(str)
    return df

df_export = sanitize_dtypes(df_proyecto)

# Mostrar columnas que cambiaron de tipo
for col in df_proyecto.columns:
    t_antes = str(df_proyecto[col].dtype)
    t_despues = str(df_export[col].dtype)
    if t_antes != t_despues:
        print(f"  {col:<30} {t_antes} -> {t_despues}")

# --- Parquet (formato principal para notebooks y ETLs) ----------------------
PARQUET_PATH = PROCESSED / "reservas_clean.parquet"
df_export.to_parquet(PARQUET_PATH, index=False, engine="pyarrow", compression="snappy")
size_parquet = PARQUET_PATH.stat().st_size / 1024
print(f"\nParquet guardado : {PARQUET_PATH}")
print(f"  Tamanio        : {size_parquet:,.1f} KB")
print(f"  Filas / Cols   : {df_export.shape[0]:,} / {df_export.shape[1]}")

# --- Excel (respaldo de auditoria y revision manual) ------------------------
EXCEL_PATH = PROCESSED / "reservas_clean.xlsx"
df_export.to_excel(EXCEL_PATH, index=False, sheet_name="reservas_clean")
size_excel = EXCEL_PATH.stat().st_size / 1024
print(f"\nExcel guardado   : {EXCEL_PATH}")
print(f"  Tamanio        : {size_excel:,.1f} KB")
print(f"  Filas / Cols   : {df_export.shape[0]:,} / {df_export.shape[1]}")

# --- Verificacion de lectura (round-trip check) -----------------------------
df_check = pd.read_parquet(PARQUET_PATH)
assert df_check.shape == df_export.shape, "ERROR: el parquet no coincide en shape"
print(f"\nVerificacion round-trip OK — {df_check.shape[0]:,} filas x {df_check.shape[1]} columnas")
print("\nDataset listo para ser consumido por:")
print("  - notebooks/03_analisis_exploratorio.ipynb")
print("  - src/etl_fact_reservas.py  (pipeline Airflow)")

df_export.dtypes.to_frame("dtype")


  edad_fue_imputada              bool -> Int8
  rango_edad                     category -> object

Parquet guardado : ..\data\processed\reservas_clean.parquet
  Tamanio        : 3,200.2 KB
  Filas / Cols   : 70,882 / 47

Excel guardado   : ..\data\processed\reservas_clean.xlsx
  Tamanio        : 14,971.4 KB
  Filas / Cols   : 70,882 / 47

Verificacion round-trip OK — 70,882 filas x 47 columnas

Dataset listo para ser consumido por:
  - notebooks/03_analisis_exploratorio.ipynb
  - src/etl_fact_reservas.py  (pipeline Airflow)


,dtype
fecha,datetime64[ns]
fllega_aco,datetime64[ns]
fsalid_aco,datetime64[ns]
fechasischin,datetime64[ns]
fcheckout,datetime64[ns]
anio,int32
mes,int32
trimestre,int32
dia_semana,object
periodo_covid,object
